# 9: Reinforcement Learning

Causal methods in reinforcement learning are actively researched.
This chapter covers their integration, existing techniques, and key challenges.

This chapter outlines current causal RL concepts and applications, noting their varied integration across RL contexts, and guides readers toward deeper study.

## 9.1: Reinforcement Learning: A Brief Introduction

Reinforcement learning (RL) is a machine learning approach where an agent learns to maximise rewards through environmental interaction.
It has achieved notable success in game-playing, such as AlphaGo Zero's superhuman performance in Go and DeepMind's DeepNash competing with human Stratego players.
RL also powers industry applications including quantitative finance, self-driving cars, and hardware design.
Additionally, RL techniques, like PPO with human feedback, are used in models such as ChatGPT to control generation quality.

In **model-based** RL, agents learn the environment's dynamics to predict state changes.
In **model-free** RL, agents learn actions directly without modelling the environment.

Online RL learns through environment interaction (trial-and-error), while offline RL uses external datasets without such interaction (<a href="#ref-levine2020offline">Levine et al. 2020</a>).

**Value-based vs. Policy-based RL**
Value-based RL approximates future rewards to select the optimal action. 
Policy-based RL directly models the probability of each action at a state.

This simplified overview of reinforcement learning assumes familiarity with causality concepts. 
Further resources are available for deeper study.

* POMDPs: observations, states, actions, rewards.
* Value-based RL: Bellman equation and Q-learning
* Policy-based RL
* Multi-armed bandits
* Visual RL environments

Key introductory resources include David Silver's DeepMind lectures, Sutton and Barto's textbook, and Hugging Face's free deep RL course.

## 9.2: Adding Causality to RL

Reinforcement learning (RL) has been successful and features inherent causal structure: agents learn how actions affect outcomes.
Causal RL integrates causal graphs and models into standard RL to better understand these action effects.

Causal inference enables reinforcement learning to incorporate domain knowledge, enhancing agent performance.
For instance, healthcare RL could use medical knowledge of drug interactions.

Key RL challenges include sample efficiency and using offline data from other agents.
Causal RL methods address both.

Elias Bareinboim's NeurIPS 2020 tutorial on Causal Reinforcement Learning covers key concepts from that period.
For deeper exploration, see his tutorial slides, notes, and videos at [crl.causalai.net](https://crl.causalai.net/).

### 9.2.1: Causality for Smarter Agents

Causal RL makes agents smarter through causal reasoning.

* Agents understanding environments using causal world models
* Adding causal bounds on regret expectations
* Improving action selection with causal knowledge
* Robust against observation or disruptions

These approaches differ in RL application but all enhance sample efficiency, optimality and agent reliability.
Future sections explore each in detail, beginning with causal knowledge in world models.

#### 9.2.1.1: Better World Models in MBRL

In model-based reinforcement learning (MBRL), *world models* learn environment dynamics.
They enable embedding domain knowledge in causal RL agents.

Li et al. (2020) propose a counterfactual "dream world" where agents simulate action outcomes using *do*-interventions, improving sample efficiency in physical tasks.
Rezende et al. (2020) offer a framework for causally accurate partial future models, useful when full future observation modelling is intractable.

Lu et al. (2022) propose *Causal Markov Decision Processes* (C-MDPs), embedding causality into state transitions and rewards within factored MDPs.
They establish a regret bound and develop factorised methods for high-dimensional state and action spaces.

Causal world models typically require explicit variables, but visual RL environments use pixel observations, necessitating *causal induction*: inferring high-level causal factors from pixel changes.
Ke et al. (2021) survey causal discovery methods for this, which can learn causal graphs during RL training.
The authors propose training an encoder (e.g., VAE) alongside a transition model (GNN or modular causal graph) using random-action trajectories.
Their modular approach best captures explicit causal structure, though its performance benefit over non-causal methods in visual RL remains unclear, as benchmarks focus on immediate action effects.

#### 9.2.1.2: Dynamic Treatment Regimes

In healthcare, reinforcement learning develops **dynamic treatment regimes (DTRs)**—adaptive care plans adjusting as a patient's condition evolves.
DTRs guide clinicians in managing chronic diseases by responding to changing health markers.
Incorporating known drug effects reduces uncertainty compared to standard methods, making causal modelling ideal for this application (Chakraborty & Murphy, 2014).

Zhang and Bareinboim developed causal dynamic treatment regimes (DTRs).
Their *UC-DTR* algorithm achieves near-optimal regret in online reinforcement learning without observational data.
*Causal UC-DTR* extends this by incorporating causal constraints on state transitions.
Both methods significantly outperform random treatment selection, with Causal UC-DTR yielding lower regret than UC-DTR.

Zhang and Bareinboim extend their 2020 causal DTR work, showing causal diagrams with structural causal models yield exponentially lower regret than non-causal RL.
They introduce two online algorithms.

* OFU-DTR applies "optimism in the face of uncertainty" to prioritise unexplored options, incorporating causal knowledge to tighten regret bounds.
* PS-DTR uses Bayesian posterior sampling with structural causal models (SCMs) to develop policies that maximise expected value from causal information.

Zhang and Bareinboim propose learning optimal dynamic treatment regimes (DTRs) from observational data using causal graphs to define treatment effects.
When causal effects are unidentifiable, they apply partial identification to derive causal bounds.
Using these bounds as a "warm start" before online learning significantly improves sample efficiency and reduces regret.

See Section 9.2.2.1 for additional work on combining online and offline data.

#### 9.2.1.3: Action Selection and Exploration

Causality aids action selection in reinforcement learning, applying to both model-based and model-free approaches.

Seitzer et al. (2021) introduce _causal influence detection_, using causal inference to identify when an agent can affect its environment (e.g., a robotic arm only influencing an object when close).
They propose _causal action influence_ (CAI), measuring causal impact via conditional mutual information in state transitions.
In robotic tasks like `FetchPickAndPlace`, CAI improves sample efficiency over standard $\epsilon$-greedy methods.
The approach requires full observability and causal variable factorisation.

Sun and Wang (2022) improved exploration in continuous control by pruning redundant actions using causal methods.
They modified the TD loss with structural causal models (SCMs) to identify relevant actions, testing by adding redundant actions to *LunarLander* (OpenAI Gym).
Their Dyn-SWAR method consistently outperformed standard TD agents.

#### 9.2.1.4: Robustness

Causality enhances RL agent resilience to environmental disruptions like blackouts or lag.
Yang et al. (2021) developed the **Causal Inference Q-Network (CIQ)**, extending DQNs to detect observational interference.
CIQ uses causal inference to reconstruct interfered states and routes modified states to the appropriate network for Q-estimation, using either a trained classifier (training) or prediction (inference).

* $State$: the unobserved condition causing observations, rewards, and interference (e.g., hardware overheating or lag).
* $Obs$: Agent's observation (always observed)
* $Int$: indicator of interference (known only at training)
* $Q$ Current reward (observed)

<img src="images/inference_dag.png" alt="" />

CIQ consistently outperformed all baselines, including standard DQNs and non-causal interference detection methods.
Standard DQNs failed with observational interference, while non-causal methods showed limited improvement but lacked reliability.
CIQ achieved target performance rapidly at 20% interference across vector and pixel environments, maintaining near-clean-input performance until ~40% interference (e.g., Cartpole).

### 9.2.2: Causality for Transfer Learning and Imitation Learning

Causal RL can combine offline data with online interaction.
Key approaches include transfer learning (merging online RL with observational data) and imitation learning (copying another agent's behaviour).
Causality helps address unknowns like a teacher agent's reward model.

#### 9.2.2.1: Transfer Learning: Combining Online and Offline Information

A key challenge in reinforcement learning is the high cost or ethical issues of real-world testing (e.g., self-driving cars or healthcare decisions).
Using existing offline data to improve agent performance (known as transfer learning) offers a solution.
However, two hurdles remain: ensuring the offline data is relevant to the task, and handling differences in how data was collected.
Causal methods address these challenges by separating and resolving them.

Forney, Pearl, and Bareinboim (2017) pioneered combining observational and experimental data using counterfactuals in a multi-armed bandit context.
They employed the *effect of the treatment on the treated* (ETT) to address unobserved confounders, fusing observational, experimental, and counterfactual data.
Their method outperformed Thompson Sampling (which uses only experimental data), achieving lower cumulative regret and higher optimal action selection rates.

Zhang and Bareinboim (2017) extend causal transfer learning to multi-armed bandits where causal effects are unidentifiable via *do*-calculus.
They derive causal bounds on expected reward distributions from observational data, using these bounds to select promising actions.
Experiments show significant efficiency gains over Thompson Sampling and UCB when causal bounds are informative.
When bounds are uninformative, performance reverts to standard methods.

Gasse et al. (2021) build on Zhang and Bareinboim's causal bounds method, combining observational and interventional data.
They extend this to MBRL world models, enhancing agents' understanding of environment dynamics in POMDPs.

Wang et al. (2021) propose *deconfounded optimistic value iteration* (DOVI), a method merging online and offline data to boost sample efficiency in reinforcement learning with confounded observational data.
DOVI uses causal inference to adjust for confounding, proving that informative offline data improves online learning efficiency—building on Zhang & Bareinboim (2017).

#### 9.2.2.2: Imitation Learning

Imitation learning differs from transfer learning by mimicking another agent's actions rather than developing a new policy.
Both use external data, but imitation focuses on behavioural replication.
Causality also offers significant benefits for handling unobservable confounders.

Zhang et al. (2020) introduced causal imitation learning, using causal graphs to define *imitability*: whether a policy can be uniquely computed from a partially observable structural causal model (POSCM).
When strict imitability fails, they propose using trajectory data alongside the causal graph.
Kumor et al. (2021) extended this framework to sequential decision-making.

### 9.2.3: Causality for Agent Explainability and Fairness

This section explores using causality to explain agent behaviour in reinforcement learning, focusing on understanding incentives, improving explainability, and enhancing fairness.

Model explainability is important in machine learning, especially for decisions in finance or healthcare.
In reinforcement learning (RL), it helps understand why an agent chooses a specific action.
Madumal et al. (2020) proposed using counterfactual explanations based on a structural causal model (SCM) to clarify RL decisions.
Tested in a StarCraft-based environment, their method was judged more trustworthy than alternatives in a user study.

Agent incentives—factors influencing an agent's decisions: differ from general explainability.
They include motivations like self-preservation or exploiting opponents.
Everitt et al. (2021) propose a framework using structural causal influence models (SCIMs) to map how observations lead to decisions and generate agent utility.
SCIMs model incentive structures through graphical relationships.

* **Materiality**: whether an observation provides important information about utility nodes in a causal influence diagram.
* **Value of Information**: Whether seeing a node before deciding benefits the agent.
  This expands materiality to include non-observable nodes.
* **Response Incentives**: whether a node affects the agent's optimal decision
  + Response Incentives relate to counterfactual fairness: a model is unfair if protected characteristics (e.g. gender, race) unduly influence its output.
  + Counterfactual fairness is covered in detail in Chapter 8.
* **Value of Control**: Agent benefits from controlling a specific node.
* Instrumental Control Incentives: whether an agent manipulates a node to gain utility.
  + D → X → U is the sole path from decision node D to utility node U, with X manipulated.

`PyCID` (Fox et al. 2021) is a Python library implementing causal influence diagrams, with SCIM framework examples in its GitHub Jupyter notebooks.

Explainability helps researchers understand an RL agent's actions, enabling fairness assessment.
Causal tools are vital for evaluating agent incentives and ensuring fairness and safety in RL.

## 9.3: Conclusions and Open Problems

This chapter explores causal inference's role in reinforcement learning (RL).
Causal insights accelerate learning, aid offline-online transfer, and clarify incentives.
Challenges include computational demands, causal identifiability issues, and high-dimensional intractability.
Despite these, causality remains valuable for boosting agent performance in RL.
